In [1]:
import time
import numpy as np
import pandas as pd
from m1_utils import *

queries, judg, corpus = load_all()
qrels = build_qrels(judg)
pids = corpus["product_id"].to_numpy()

print("queries:", queries.shape, "| judgments:", judg.shape, "| corpus:", corpus.shape)
print(queries["role"].value_counts())
assert set(judg["product_id"]) <= set(pids)
assert set(queries["query_id"]) == set(qrels)

dev = queries[queries["role"] == "dev"].reset_index(drop=True)
test = queries[queries["role"] == "eval_test"].reset_index(drop=True)
print("dev:", len(dev), "| test:", len(test), "| test small_version:", int(test["small"].sum()))

queries: (5664, 12) | judgments: (100724, 8) | corpus: (51318, 7)
role
train_other    3317
eval_test      1295
dev             500
clf_train       386
clf_test         83
clf_val          83
Name: count, dtype: int64
dev: 500 | test: 1295 | test small_version: 507


In [2]:
from rank_bm25 import BM25Okapi

t0 = time.time()
tok_docs = [tokenize(t) for t in corpus["doc_text"]]
bm25 = BM25Okapi(tok_docs)            # default k1=1.5, b=0.75
print(f"index built in {time.time()-t0:.1f}s | docs: {len(tok_docs)}")

t0 = time.time()
ranked = {}
sample = dev.head(20)
for qid, qtext in zip(sample["query_id"], sample["query"]):
    s = bm25.get_scores(tokenize(qtext))
    ranked[qid] = list(pids[np.argsort(-s)[:10]])
print(f"20 queries in {time.time()-t0:.1f}s")
print(evaluate(ranked, qrels).mean().round(4))

index built in 4.1s | docs: 51318
20 queries in 1.8s
ndcg@10    0.2081
mrr@10     0.3000
dtype: float64


In [3]:
eval_q = pd.concat([dev, test]).reset_index(drop=True)
eval_q[["query_id", "query", "role", "small", "source", "scope_group"]].to_parquet(OUT / "eval_query_order.parquet")

t0 = time.time()
bm25_scores = np.zeros((len(eval_q), len(pids)), dtype=np.float32)
for i, qtext in enumerate(eval_q["query"]):
    bm25_scores[i] = bm25.get_scores(tokenize(qtext))
    if i % 300 == 0:
        print(i, f"{time.time()-t0:.0f}s")
np.save(OUT / "bm25_scores.npy", bm25_scores)
print("done", bm25_scores.shape, f"{time.time()-t0:.0f}s")

def ranked_from_matrix(S, rows, k=10):
    out = {}
    for i in rows:
        top = np.argpartition(-S[i], k)[:k]
        top = top[np.argsort(-S[i][top])]
        out[eval_q.at[i, "query_id"]] = list(pids[top])
    return out

dev_rows = eval_q.index[eval_q["role"] == "dev"]
res_bm25 = evaluate(ranked_from_matrix(bm25_scores, dev_rows), qrels)
print("\nCondition 1 BM25 (dev):")
print(res_bm25.mean().round(4))
print(res_bm25.join(eval_q.set_index("query_id")[["source"]]).groupby("source").agg(["mean", "count"]).round(3))

0 0s
300 23s
600 45s
900 72s
1200 94s
1500 118s
done (1795, 51318) 147s

Condition 1 BM25 (dev):
ndcg@10    0.2503
mrr@10     0.3806
dtype: float64
              ndcg@10       mrr@10      
                 mean count   mean count
source                                  
behavioral      0.000     1  0.000     1
negations       0.050     9  0.050     9
other           0.255   477  0.389   477
parse_pattern   0.245    13  0.327    13


In [4]:
import torch, faiss
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
model.max_seq_length = 256

t0 = time.time()
doc_emb = model.encode(corpus["doc_text"].tolist(), batch_size=128,
                       normalize_embeddings=True, show_progress_bar=True,
                       convert_to_numpy=True).astype("float32")
print("encoded", doc_emb.shape, f"{time.time()-t0:.0f}s")
np.save(OUT / "doc_emb_minilm.npy", doc_emb)

index = faiss.IndexFlatIP(doc_emb.shape[1])   # exact search; 51k x 384 is tiny
index.add(doc_emb)
faiss.write_index(index, str(OUT / "faiss_minilm.index"))
print("index size:", index.ntotal)

E:\GitHub\j26-ds-358\member1_retrieval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu


Batches: 100%|██████████| 401/401 [42:20<00:00,  6.34s/it]


encoded (51318, 384) 2542s
index size: 51318


In [5]:
q_emb = model.encode(eval_q["query"].tolist(), batch_size=128,
                     normalize_embeddings=True, convert_to_numpy=True).astype("float32")
np.save(OUT / "query_emb_raw_minilm.npy", q_emb)

dense_raw_scores = q_emb @ doc_emb.T
np.save(OUT / "dense_raw_scores.npy", dense_raw_scores.astype("float32"))

# cross-check: FAISS top-10 must match the matrix top-10
D, I = index.search(q_emb[dev_rows.to_numpy()], 10)
mat_top = [set(pids[np.argsort(-dense_raw_scores[i])[:10]]) for i in dev_rows]
faiss_top = [set(pids[row]) for row in I]
print("FAISS vs matrix top-10 overlap:", np.mean([len(a & b) / 10 for a, b in zip(mat_top, faiss_top)]).round(4))

res_dense = evaluate(ranked_from_matrix(dense_raw_scores, dev_rows), qrels)
cmp = pd.DataFrame({"C1 BM25": res_bm25.mean(), "C2 raw dense": res_dense.mean()})
print(cmp.round(4))

src = eval_q.set_index("query_id")["source"]
by_src = pd.concat({"C1 BM25": res_bm25["ndcg@10"], "C2 raw dense": res_dense["ndcg@10"]}, axis=1).join(src).groupby("source").mean().round(3)
print(by_src)

FAISS vs matrix top-10 overlap: 0.9998
         C1 BM25  C2 raw dense
ndcg@10   0.2503        0.2347
mrr@10    0.3806        0.3727
               C1 BM25  C2 raw dense
source                              
behavioral       0.000         0.385
negations        0.050         0.047
other            0.255         0.235
parse_pattern    0.245         0.329


In [6]:
# candidate pool = union of top-100 from each retriever, min-max normalised inside the pool
K = 100
alphas = np.round(np.linspace(0, 1, 11), 1)
dev_rows = eval_q.index[eval_q["role"] == "dev"]

def cand_blend(i, K=100):
    b, d = bm25_scores[i], dense_raw_scores[i]
    cand = np.union1d(np.argpartition(-b, K)[:K], np.argpartition(-d, K)[:K])
    return cand, minmax(b[cand]), minmax(d[cand])

pre = {i: cand_blend(i) for i in dev_rows}

grid = {}
for a in alphas:
    ranked = {}
    for i in dev_rows:
        cand, bn, dn = pre[i]
        s = a * bn + (1 - a) * dn
        ranked[eval_q.at[i, "query_id"]] = list(pids[cand[np.argsort(-s)[:10]]])
    grid[a] = evaluate(ranked, qrels)["ndcg@10"]
G = pd.DataFrame(grid)            # rows = dev queries, columns = alpha (1.0 = BM25, 0.0 = dense)

print("mean NDCG@10 by fixed alpha:")
print(G.mean().round(4).to_string())
best_a = G.mean().idxmax()
print(f"\nbest fixed alpha = {best_a} -> {G[best_a].mean():.4f}")
print(f"oracle per-query alpha   -> {G.max(axis=1).mean():.4f}   (upper bound for any alpha predictor)")

bm_better = (G[1.0] > G[0.0]).mean(); de_better = (G[0.0] > G[1.0]).mean()
print(f"\nBM25 strictly better: {bm_better:.1%} | dense strictly better: {de_better:.1%} | tie: {1-bm_better-de_better:.1%}")

# crude Exact-like proxy: query has a token mixing letters and digits (model code)
qtext = eval_q.loc[dev_rows, "query"].to_numpy()
has_model = np.array([any(re.search(r"[a-z]", w) and re.search(r"\d", w) for w in re.findall(r"[a-z0-9-]+", t.lower())) for t in qtext])
print("\nby model-token proxy (mean NDCG@10):")
print(pd.DataFrame({"alpha=1 (BM25)": G[1.0].values, "alpha=0 (dense)": G[0.0].values,
                    f"alpha={best_a} (fixed)": G[best_a].values, "oracle": G.max(axis=1).values,
                    "model_token": has_model}).groupby("model_token").agg(["mean"]).round(3).assign(n=pd.Series(has_model).value_counts()))

mean NDCG@10 by fixed alpha:
0.0    0.2347
0.1    0.2597
0.2    0.2771
0.3    0.2911
0.4    0.2950
0.5    0.2911
0.6    0.2864
0.7    0.2751
0.8    0.2698
0.9    0.2599
1.0    0.2501

best fixed alpha = 0.4 -> 0.2950
oracle per-query alpha   -> 0.3669   (upper bound for any alpha predictor)

BM25 strictly better: 42.0% | dense strictly better: 39.8% | tie: 18.2%

by model-token proxy (mean NDCG@10):
            alpha=1 (BM25) alpha=0 (dense) alpha=0.4 (fixed) oracle    n
                      mean            mean              mean   mean     
model_token                                                             
False                 0.23           0.223             0.275  0.344  424
True                  0.36           0.300             0.409  0.497   76


In [7]:
mt_idx = np.where(has_model)[0]; nm_idx = np.where(~has_model)[0]
curve = pd.DataFrame({"model_token (n=%d)" % len(mt_idx): G.iloc[mt_idx].mean(),
                      "no_model_token (n=%d)" % len(nm_idx): G.iloc[nm_idx].mean()}).round(3)
print(curve.to_string())
print("\nbest alpha per group:", curve.idxmax().to_dict())

     model_token (n=76)  no_model_token (n=424)
0.0               0.300                   0.223
0.1               0.336                   0.246
0.2               0.376                   0.259
0.3               0.398                   0.272
0.4               0.409                   0.275
0.5               0.399                   0.272
0.6               0.400                   0.266
0.7               0.387                   0.255
0.8               0.382                   0.250
0.9               0.373                   0.240
1.0               0.360                   0.230

best alpha per group: {'model_token (n=76)': 0.4, 'no_model_token (n=424)': 0.4}
